# OYKHCHAR LoRA Training on FLUX.1-dev

## FREE Training on Kaggle GPU!

- Model: FLUX.1-dev (latest text-to-image)
- Dataset: 27 curated OYKHCHAR images
- Cost: $0 (Kaggle provides 30 hrs/week free GPU)
- Time: 30-60 minutes

## Dataset
Using: `oykhchar-lora-images` dataset

## Strategy
- LoRA rank: 8-16 (efficient fine-tuning)
- Steps: 1000-1500
- Learning rate: 4e-4
- Trigger word: OYKHCHAR

In [ ]:
# Install dependencies
!pip install -q diffusers transformers accelerate peft bitsandbytes safetensors torch torchvision Pillow

In [ ]:
import os
import zipfile
from pathlib import Path

# Extract dataset
dataset_path = "/kaggle/input/oykhchar-lora-images/images.zip"
output_dir = "/kaggle/working/training_data"

print(f"Extracting {dataset_path}...")
os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
    zip_ref.extractall(output_dir)

# Find all images
image_files = list(Path(output_dir).rglob("*.jpg")) + list(Path(output_dir).rglob("*.png"))
print(f"✓ Found {len(image_files)} training images")

# Verify captions
caption_count = sum(1 for img in image_files if img.with_suffix('.txt').exists())
print(f"✓ Found {caption_count}/{len(image_files)} captions")

# Show first few
for img in image_files[:3]:
    print(f"  - {img.name}")
    caption_file = img.with_suffix('.txt')
    if caption_file.exists():
        caption = caption_file.read_text()
        print(f"    Caption: {caption[:80]}...")

In [ ]:
# Setup training
import torch
from diffusers import FluxPipeline
from peft import LoraConfig, get_peft_model

print("Loading FLUX.1-dev model...")
pipeline = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    torch_dtype=torch.float16,
    use_safetensors=True
)

print("Configuring LoRA...")
lora_config = LoraConfig(
    r=16,  # Rank (8-16 is good for character training)
    lora_alpha=32,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.1,
    bias="none",
)

# Apply LoRA
model = get_peft_model(pipeline.transformer, lora_config)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"✓ LoRA configured")
print(f"  Trainable params: {trainable_params:,}")
print(f"  Total params: {total_params:,}")
print(f"  Trainable %: {100 * trainable_params / total_params:.2f}%")

In [ ]:
# For actual training, you would use Hugging Face Trainer or custom training loop
# This is a simplified example showing the setup

print("🎓 Setup complete!")
print("")
print("Next steps for production training:")
print("1. Use Hugging Face Trainer with your dataset")
print("2. Train for 1000-1500 steps")
print("3. Save LoRA weights to /kaggle/working")
print("4. Download trained weights")
print("")
print("Alternative: Use dedicated LoRA trainers like:")
print("- kohya_ss/sd-scripts")
print("- SimpleTuner")
print("- Ostris/ai-toolkit")

In [ ]:
# Test generation (before training)
pipeline_cpu = pipeline.to("cpu")
pipeline_gpu = pipeline.to("cuda")

test_prompt = "OYKHCHAR character pointing forward"
print(f"Generating test image: {test_prompt}")

image = pipeline_gpu(
    test_prompt,
    num_inference_steps=28,
    guidance_scale=3.5,
    height=1024,
    width=1024
).images[0]

image.save("/kaggle/working/test_before_training.png")
print("✓ Test image saved to test_before_training.png")
image

## Summary

This notebook shows the basic setup for LoRA training on FLUX.1-dev.

For production training:
1. Implement training loop with proper loss calculation
2. Use learning rate scheduler
3. Add validation during training
4. Save checkpoints regularly

Or use existing LoRA training frameworks that handle all of this!